In [1]:
from training.sl import load_model
from models.glimpse import TSPTransformer

model = load_model(TSPTransformer, model_name="model_glimpse")

In [2]:
import torch
import random
from settings import MODELS_FOLDER
from instances.instances import read_instances
from models.glimpse import TSPTransformer
from data.adapters.input.basic import BasicInputAdapter
from training.rl import train_pomo_rl, POMOConfig, load_model

def main():
    device = torch.device("cuda" if torch.cuda.is_available() 
                          else "mps" if torch.backends.mps.is_available() 
                          else "cpu")
    print(f"** Iniciando entrenamiento POMO en: {device}")

    # 1. Cargar el único archivo de instancias
    print("** Cargando instancias...")
    all_instances = read_instances("TSP50.pkl")
    
    # 2. Hacer el split (Train y Dev)
    # Mezclamos aleatoriamente para garantizar que no haya sesgo de orden
    random.seed(42)
    random.shuffle(all_instances)
    
    # Por ejemplo, usamos el 90% para entrenar y 10% para validar
    split_idx = int(len(all_instances) * 0.99)
    train_instances = all_instances[:split_idx]
    val_instances = all_instances[split_idx:]
    
    print(f"Instancias totales: {len(all_instances)} "
          f"(Train: {len(train_instances)}, Dev: {len(val_instances)})")

    # 3. Configuración del Adaptador
    MAX_CITIES = 50
    input_adapter_config = (BasicInputAdapter, MAX_CITIES)

    # 4. Inicializar el Modelo
    model = load_model(TSPTransformer, model_name="model_glimpse")

    # 5. Inicializar el Optimizador
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

    # 6. Configuración POMO
    pomo_config = POMOConfig(
        updates=100,              
        k_rollouts=16,             
        instances_per_update=8,   
        adv_clip=4.0,              
        grad_clip=1.0,             
        minibatch_size=64,       
        eval_interval=5,          
        patience=20                
    )

    # 7. Ejecutar Entrenamiento
    save_directory = MODELS_FOLDER / "pomo_tsp_experiment"
    
    train_pomo_rl(
        model=model,
        optimizer=optimizer,
        train_instances=train_instances,
        val_instances=val_instances,
        pomo_config=pomo_config,
        input_adapter_config=input_adapter_config,
        device=device,
        save_dir=save_directory,
        model_name="best_pomo_model.pth"
    )

if __name__ == "__main__":
    main()

** Iniciando entrenamiento POMO en: cpu
** Cargando instancias...
Instancias totales: 1000 (Train: 990, Dev: 10)
Iniciando POMO RL por 100 updates...
Update 00001 | PG: -0.088 | Ent: 1.614 | AvgCost: 10.23 | 269 steps/s
Update 00002 | PG: -0.094 | Ent: 1.574 | AvgCost: 9.83 | 263 steps/s
Update 00003 | PG: -0.098 | Ent: 1.555 | AvgCost: 9.87 | 256 steps/s
Update 00004 | PG: -0.093 | Ent: 1.562 | AvgCost: 9.60 | 272 steps/s
Update 00005 | PG: -0.092 | Ent: 1.530 | AvgCost: 9.72 | 263 steps/s
   >>> Evaluación Val Set: 7.32 | (Mejor: 7.32)
   >>> Mejor modelo actualizado.
Update 00006 | PG: -0.095 | Ent: 1.518 | AvgCost: 9.69 | 264 steps/s
Update 00007 | PG: -0.094 | Ent: 1.520 | AvgCost: 9.39 | 268 steps/s
Update 00008 | PG: -0.093 | Ent: 1.507 | AvgCost: 9.43 | 265 steps/s
Update 00009 | PG: -0.077 | Ent: 1.526 | AvgCost: 9.54 | 268 steps/s
Update 00010 | PG: -0.104 | Ent: 1.500 | AvgCost: 9.48 | 272 steps/s
   >>> Evaluación Val Set: 6.93 | (Mejor: 6.93)
   >>> Mejor modelo actualizad

In [5]:
from training.sl import load_model
from models.glimpse import TSPTransformer

model = load_model(TSPTransformer, model_name="best_pomo_model")

In [6]:
from solvers.eval import evaluate

adapter_config = (BasicInputAdapter, 50)
sols = evaluate(model, "benchmarks/B50.pkl", input_adapter_config=adapter_config, num_workers=12)

Iniciando evaluación de 100 instancias con 12 workers...
** Evaluación completada. Costo promedio: 6.6539
